# ModernBERT pairwise con secuencias largas

En esta sección se evalúa ModernBERT manteniendo la misma metodología pairwise, las mismas particiones train-test y los mismos folds utilizados en el experimento anterior con BERT.

El objetivo es estudiar si permitir secuencias de mayor longitud mejora la selección de la respuesta top-1. Esta comparación resulta especialmente relevante porque el análisis realizado con bert-base-uncased mostró que un 29,34 % de los ejemplos superaba el límite de 512 tokens.

Para aislar el efecto del cambio de modelo y de la longitud máxima de entrada, se mantiene la semilla 27 y no se modifican ni la construcción de pares ni el procedimiento de evaluación mediante torneo pairwise.

In [ ]:
import torch
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))


True
NVIDIA A40


In [ ]:
import numpy as np
import pandas as pd

from sklearn.model_selection import (
    train_test_split,
    StratifiedKFold
)

In [ ]:
url = (
    "https://huggingface.co/datasets/nbertagnolli/counsel-chat/"
    "raw/main/20220401_counsel_chat.csv"
)

df = pd.read_csv(url)

print("Preguntas totales:", df["questionID"].nunique())
print("Respuestas totales:", len(df))

Preguntas totales: 940
Respuestas totales: 2775


#### Preparación de los datos

Se conservan únicamente las preguntas con al menos dos respuestas candidatas y con alguna señal de preferencia, es decir, aquellas en las que al menos una respuesta tiene algún upvote.

In [ ]:
df_modelo = df.copy()

# Preguntas con al menos dos respuestas
n_respuestas = (
    df_modelo.groupby("questionID")["answerText"]
    .transform("size")
)

df_modelo = df_modelo[
    n_respuestas >= 2
].copy()

# Preguntas con alguna señal de preferencia
max_upvotes = (
    df_modelo.groupby("questionID")["upvotes"]
    .transform("max")
)

df_modelo = df_modelo[
    max_upvotes > 0
].copy()

# Target: respuesta con máximo número de upvotes
df_modelo["target"] = (
    df_modelo["upvotes"]
    == df_modelo.groupby("questionID")["upvotes"]
    .transform("max")
).astype(int)

print(
    "Preguntas incluidas:",
    df_modelo["questionID"].nunique()
)

print(
    "Respuestas incluidas:",
    len(df_modelo)
)

Preguntas incluidas: 276
Respuestas incluidas: 1732


#### Caracterización de las preguntas

Para construir particiones más comparables, se resume la información a nivel de pregunta.
Para cada questionID se calcula:

- El número de respuestas candidatas.
- Cuántas respuestas comparten el máximo número de upvotes.
- si existe o no empate en la respuesta top.

In [ ]:
info_preguntas = (
    df_modelo.groupby("questionID")
    .agg(
        n_respuestas=("answerText", "size")
    )
    .reset_index()
)

n_top = (
    df_modelo.assign(
        es_top=lambda x:
            x["upvotes"]
            == x.groupby("questionID")["upvotes"].transform("max")
    )
    .groupby("questionID")["es_top"]
    .sum()
    .reset_index(name="n_respuestas_top")
)

info_preguntas = info_preguntas.merge(
    n_top,
    on="questionID"
)

info_preguntas["empate"] = (
    info_preguntas["n_respuestas_top"] > 1
)

display(info_preguntas.head())

,questionID,n_respuestas,n_respuestas_top,empate
0,0,86,1,False
1,1,5,1,False
2,2,2,1,False
3,3,3,1,False
4,4,6,1,False


Comprobación rápida para detectar si algo salió raro:

In [ ]:
print("Preguntas:", len(info_preguntas))
print(
    "Preguntas con empate:",
    info_preguntas["empate"].sum()
)
print(
    "Porcentaje con empate:",
    f"{info_preguntas['empate'].mean():.2%}"
)

Preguntas: 276
Preguntas con empate: 62
Porcentaje con empate: 22.46%


#### Definición de los estratos

Las preguntas se agrupan según dos características:

- El número de respuestas candidatas.
- La presencia o ausencia de empate en el máximo número de upvotes.

Para el número de respuestas se utilizan los intervalos 2, 3, 4-5, 6-10 y >10.
La combinación de ambas características define el estrato de cada pregunta.
Estos estratos se utilizarán posteriormente para intentar mantener una composición similar entre train, test y los folds de validación.

In [ ]:
def grupo_n_respuestas(n):
    if n == 2:
        return "2"
    elif n == 3:
        return "3"
    elif n <= 5:
        return "4-5"
    elif n <= 10:
        return "6-10"
    else:
        return ">10"


info_preguntas["grupo_respuestas"] = (
    info_preguntas["n_respuestas"]
    .apply(grupo_n_respuestas)
)

info_preguntas["estrato"] = (
    info_preguntas["grupo_respuestas"]
    + ", empate="
    + info_preguntas["empate"].astype(str)
)

Se comprueba el número de preguntas de cada estrato antes de realizar las divisiones. Esto permite verificar que los grupos contienen suficientes ejemplos para poder distribuirlos entre las distintas particiones.

In [ ]:
distribucion_estratos = (
    info_preguntas["estrato"]
    .value_counts()
    .sort_index()
)

display(
    distribucion_estratos.to_frame(
        "Número de preguntas"
    )
)

,Número de preguntas
estrato,
"2, empate=False",82
"2, empate=True",16
"3, empate=False",42
"3, empate=True",10
"4-5, empate=False",35
"4-5, empate=True",10
"6-10, empate=False",34
"6-10, empate=True",14
">10, empate=False",21


## Split train-test estratificado

Se reproduce exactamente la división del notebook 04 (misma estratificación y semilla 27), de modo que BERT y ModernBERT se evalúan sobre las mismas preguntas.

In [ ]:
from sklearn.model_selection import train_test_split

preguntas_train, preguntas_test = train_test_split(
    info_preguntas,
    test_size=0.20,
    random_state=27,
    stratify=info_preguntas["estrato"]
)

Una vez asignadas las preguntas a train y test, se recuperan del dataset original todas las respuestas correspondientes a esas preguntas. Así, cada pregunta mantiene juntas todas sus respuestas dentro de una única partición.

In [ ]:
train_equilibrado = df_modelo[
    df_modelo["questionID"].isin(
        preguntas_train["questionID"]
    )
].copy()

test_equilibrado = df_modelo[
    df_modelo["questionID"].isin(
        preguntas_test["questionID"]
    )
].copy()

print(
    "Preguntas en train:",
    train_equilibrado["questionID"].nunique()
)

print(
    "Preguntas en test:",
    test_equilibrado["questionID"].nunique()
)

Preguntas en train: 220
Preguntas en test: 56


In [ ]:
preguntas_comunes = (
    set(train_equilibrado["questionID"])
    & set(test_equilibrado["questionID"])
)

print(
    "Preguntas compartidas entre train y test:",
    len(preguntas_comunes)
)

Preguntas compartidas entre train y test: 0


No existe solapamiento de preguntas entre train y test, por lo que todas las respuestas asociadas a un mismo questionID permanecen dentro de una única partición.

## Construcción de los 5 folds estratificados

Se utilizan los mismos folds que en el notebook 04, construidos con StratifiedKFold sobre las preguntas de train.


In [ ]:
from sklearn.model_selection import StratifiedKFold

skf = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=27
)

folds_preguntas = list(
    skf.split(
        preguntas_train,
        preguntas_train["estrato"]
    )
)

print(
    "Número de folds:",
    len(folds_preguntas)
)

Número de folds: 5


Los 5 folds ya están definidos por pregunta. Ahora se localizan en train_equilibrado todas las respuestas correspondientes a las preguntas de entrenamiento y validación de cada fold.

In [ ]:
cv_splits_equilibrados = []

for train_q_idx, val_q_idx in folds_preguntas:

    ids_train_fold = set(
        preguntas_train.iloc[
            train_q_idx
        ]["questionID"]
    )

    ids_val_fold = set(
        preguntas_train.iloc[
            val_q_idx
        ]["questionID"]
    )

    train_idx = np.where(
        train_equilibrado["questionID"]
        .isin(ids_train_fold)
    )[0]

    val_idx = np.where(
        train_equilibrado["questionID"]
        .isin(ids_val_fold)
    )[0]

    cv_splits_equilibrados.append(
        (train_idx, val_idx)
    )

Comprobación:

In [ ]:
for fold, (train_idx, val_idx) in enumerate(
    cv_splits_equilibrados,
    start=1
):

    train_fold = train_equilibrado.iloc[
        train_idx
    ]

    val_fold = train_equilibrado.iloc[
        val_idx
    ]

    comunes = (
        set(train_fold["questionID"])
        & set(val_fold["questionID"])
    )

    print(
        f"Fold {fold}: "
        f"{train_fold['questionID'].nunique()} preguntas train | "
        f"{val_fold['questionID'].nunique()} preguntas validación | "
        f"{len(comunes)} compartidas"
    )

Fold 1: 176 preguntas train | 44 preguntas validación | 0 compartidas
Fold 2: 176 preguntas train | 44 preguntas validación | 0 compartidas
Fold 3: 176 preguntas train | 44 preguntas validación | 0 compartidas
Fold 4: 176 preguntas train | 44 preguntas validación | 0 compartidas
Fold 5: 176 preguntas train | 44 preguntas validación | 0 compartidas


Se comprueba que en los 5 folds hay 0 preguntas compartidas entre entrenamiento y validación.

## Comprobación de las particiones

Antes del entrenamiento de ModernBERT, se analiza la composición de los folds de validación y del conjunto de test.
Se comprueba:
- El número de preguntas.
- La proporción de preguntas con empate.
- El número medio y mediano de respuestas candidatas.
- La distribución del número de respuestas por pregunta.

El objetivo es verificar que las particiones sean aproximadamente comparables antes de utilizar sus resultados para evaluar el modelo.

In [ ]:
def resumen_particion(info, nombre):
    return {
        "Partición": nombre,
        "Preguntas": len(info),
        "Preguntas con empate": info["empate"].sum(),
        "% con empate": info["empate"].mean() * 100,
        "Media respuestas": info["n_respuestas"].mean(),
        "Mediana respuestas": info["n_respuestas"].median(),
        "Mín. respuestas": info["n_respuestas"].min(),
        "Máx. respuestas": info["n_respuestas"].max()
    }

In [ ]:
resumen_nuevo = []

for fold, (_, val_q_idx) in enumerate(
    folds_preguntas,
    start=1
):

    info_val = preguntas_train.iloc[
        val_q_idx
    ]

    resumen_nuevo.append(
        resumen_particion(
            info_val,
            f"Fold {fold}"
        )
    )

resumen_nuevo.append(
    resumen_particion(
        preguntas_test,
        "Test"
    )
)

resumen_nuevo = pd.DataFrame(
    resumen_nuevo
)

display(
    resumen_nuevo.round(2)
)

,Partición,Preguntas,Preguntas con empate,% con empate,Media respuestas,Mediana respuestas,Mín. respuestas,Máx. respuestas
0,Fold 1,44,10,22.73,6.11,3.0,2,51
1,Fold 2,44,9,20.45,5.25,3.0,2,28
2,Fold 3,44,9,20.45,5.52,3.0,2,31
3,Fold 4,44,11,25.00,8.61,3.0,2,107
4,Fold 5,44,11,25.00,6.93,3.0,2,86
5,Test,56,12,21.43,5.45,3.0,2,35


**Número de preguntas:** Los cinco folds tienen 44 preguntas cada uno y test tiene 56. La proporción de preguntas con empate es similar en todas las particiones, igual que en el notebook 04.





#### Distribución del número de respuestas

In [ ]:
distribuciones_nuevas = []

for fold, (_, val_q_idx) in enumerate(
    folds_preguntas,
    start=1
):

    info_val = preguntas_train.iloc[
        val_q_idx
    ]

    dist = (
        info_val["grupo_respuestas"]
        .value_counts(normalize=True)
        .mul(100)
    )

    dist.name = f"Fold {fold}"
    distribuciones_nuevas.append(dist)


dist_test = (
    preguntas_test["grupo_respuestas"]
    .value_counts(normalize=True)
    .mul(100)
)

dist_test.name = "Test"

distribuciones_nuevas.append(
    dist_test
)

tabla_distribucion = pd.concat(
    distribuciones_nuevas,
    axis=1
).fillna(0)

display(
    tabla_distribucion
    .reindex(
        ["2", "3", "4-5", "6-10", ">10"]
    )
    .round(2)
)

,Fold 1,Fold 2,Fold 3,Fold 4,Fold 5,Test
grupo_respuestas,,,,,,
2,36.36,34.09,34.09,36.36,36.36,35.71
3,18.18,20.45,20.45,18.18,15.91,19.64
4-5,18.18,15.91,15.91,15.91,15.91,16.07
6-10,15.91,18.18,18.18,15.91,18.18,17.86
>10,11.36,11.36,11.36,13.64,13.64,10.71


La distribución del número de respuestas es similar entre los folds y test, igual que en el notebook 04.



#### Distribución de los estratos

Esta comprobación permite ver conjuntamente las dos variables utilizadas para equilibrar las particiones: el número de respuestas candidatas y la presencia o ausencia de empate en el máximo de upvotes. Al tratarse de las mismas particiones, la distribución coincide con la del notebook 04.

In [ ]:
distribuciones_estratos = []

for fold, (_, val_q_idx) in enumerate(
    folds_preguntas,
    start=1
):

    info_val = preguntas_train.iloc[
        val_q_idx
    ]

    dist = (
        info_val["estrato"]
        .value_counts(normalize=True)
        .mul(100)
    )

    dist.name = f"Fold {fold}"
    distribuciones_estratos.append(dist)


dist_test_estratos = (
    preguntas_test["estrato"]
    .value_counts(normalize=True)
    .mul(100)
)

dist_test_estratos.name = "Test"

distribuciones_estratos.append(
    dist_test_estratos
)

tabla_estratos = pd.concat(
    distribuciones_estratos,
    axis=1
).fillna(0)

display(
    tabla_estratos
    .sort_index()
    .round(2)
)

,Fold 1,Fold 2,Fold 3,Fold 4,Fold 5,Test
estrato,,,,,,
"2, empate=False",29.55,29.55,29.55,29.55,29.55,30.36
"2, empate=True",6.82,4.55,4.55,6.82,6.82,5.36
"3, empate=False",15.91,15.91,15.91,13.64,13.64,16.07
"3, empate=True",2.27,4.55,4.55,4.55,2.27,3.57
"4-5, empate=False",13.64,13.64,13.64,11.36,11.36,12.50
"4-5, empate=True",4.55,2.27,2.27,4.55,4.55,3.57
"6-10, empate=False",11.36,13.64,13.64,11.36,11.36,12.50
"6-10, empate=True",4.55,4.55,4.55,4.55,6.82,5.36
">10, empate=False",6.82,6.82,6.82,9.09,9.09,7.14


La distribución de los estratos es muy similar entre los cinco folds y test. Entre folds, cada estrato difiere como mucho en una pregunta. Test tampoco puede tener exactamente los mismos porcentajes que los folds, porque tiene un número distinto de preguntas (56 frente a 44), y por eso aparecen pequeñas diferencias entre ellos. La tabla es idéntica a la del notebook 04, ya que las particiones son las mismas. Esto garantiza que BERT y ModernBERT se entrenan y evalúan exactamente sobre las mismas preguntas, de modo que las diferencias de rendimiento entre ambos pueden atribuirse al modelo y no a la partición.



## ModernBERT pairwise con las particiones del notebook 04

A continuación se repite el experimento pairwise utilizando answerdotai/ModernBERT-base.

Se mantienen las mismas particiones, construcción de pares, hiperparámetros y procedimiento de evaluación empleados con BERT. La única diferencia de configuración es la longitud máxima de entrada, que pasa de 512 a 2048 tokens (capacidad para procesar secuencias más largas).


#### Preparación de ModernBERT

Se carga answerdotai/ModernBERT-base y se comprueba que PyTorch puede utilizar la GPU disponible.

Se mantiene la semilla 27 para favorecer la reproducibilidad del experimento.

Antes de fijar la longitud utilizada durante el entrenamiento, se analizará la distribución real de tokens de los pares del conjunto de entrenamiento.

In [ ]:
import gc
import torch

from itertools import combinations
from datasets import Dataset

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    DataCollatorWithPadding,
    TrainingArguments,
    Trainer,
    set_seed
)

set_seed(27)

print("GPU disponible:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

/home/jpacurio/tfg_env/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


GPU disponible: True
GPU: NVIDIA A40


In [ ]:
MODEL_NAME = "answerdotai/ModernBERT-base"

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME
)

data_collator = DataCollatorWithPadding(
    tokenizer=tokenizer
)

print("Modelo:", MODEL_NAME)
print("Longitud máxima soportada:", tokenizer.model_max_length)

Modelo: answerdotai/ModernBERT-base
Longitud máxima soportada: 8192


El modelo admite hasta 8192 tokens.

#### Construcción de los pares de entrenamiento

La construcción de ejemplos pairwise se mantiene idéntica al experimento con BERT.

Para cada pregunta se comparan respuestas con distinto número de upvotes. Los pares empatados se descartan y cada comparación se incorpora en ambos órdenes para evitar introducir un sesgo asociado a la posición de la respuesta.

La etiqueta es:

- 0 si la respuesta A es la preferida.

- 1 si la respuesta B es la preferida.

In [ ]:
def crear_pares_entrenamiento(datos):
    ejemplos = []

    for question_id, grupo in datos.groupby("questionID"):

        filas = list(
            grupo.itertuples(index=False)
        )

        for respuesta_1, respuesta_2 in combinations(
            filas, 2
        ):

            # No existe una preferencia clara
            if respuesta_1.upvotes == respuesta_2.upvotes:
                continue

            pregunta = str(
                respuesta_1.questionText
            )

            texto_1 = str(
                respuesta_1.answerText
            )

            texto_2 = str(
                respuesta_2.answerText
            )

            if respuesta_1.upvotes > respuesta_2.upvotes:
                label = 0
            else:
                label = 1


            # Orden A-B
            ejemplos.append({
                "texto_a": (
                    f"Question: {pregunta}\n"
                    f"Response A: {texto_1}"
                ),
                "texto_b": (
                    f"Response B: {texto_2}"
                ),
                "label": label
            })

            # Orden B-A
            ejemplos.append({
                "texto_a": (
                    f"Question: {pregunta}\n"
                    f"Response A: {texto_2}"
                ),
                "texto_b": (
                    f"Response B: {texto_1}"
                ),
                "label": 1 - label
            })

    return pd.DataFrame(ejemplos)

#### Análisis de longitud con ModernBERT

Antes del entrenamiento se calcula la longitud de los pares utilizando el tokenizer de ModernBERT.

Este análisis permite comprobar cuántos ejemplos superarían los 512 tokens utilizados anteriormente con BERT y si el nuevo límite de 2048 tokens evita el truncado en la mayor parte o totalidad de los pares.

In [ ]:
pares_analisis_modernbert = crear_pares_entrenamiento(
    train_equilibrado
)

def longitud_tokens_pares(pares):
    longitudes = []

    for texto_a, texto_b in zip(
        pares["texto_a"],
        pares["texto_b"]
    ):
        tokens = tokenizer(
            texto_a,
            texto_b,
            truncation=False
        )

        longitudes.append(
            len(tokens["input_ids"])
        )

    return np.array(longitudes)


longitudes_modernbert = longitud_tokens_pares(
    pares_analisis_modernbert
)

In [ ]:
MAX_LENGTH = 2048

print("Número de ejemplos (parejas en ambos órdenes):", len(longitudes_modernbert))


print(
    "Ejemplos con más de 512 tokens:",
    (longitudes_modernbert > 512).sum()
)

print(
    "Porcentaje con más de 512 tokens:",
    f"{(longitudes_modernbert > 512).mean():.2%}"
)


print(
    "Longitud mediana:",
    np.median(longitudes_modernbert)
)

print(
    "Longitud máxima:",
    longitudes_modernbert.max()
)

print(
    "Ejemplos con más de 2048 tokens:",
    (longitudes_modernbert > MAX_LENGTH).sum()
)

print(
    "Porcentaje truncado con 2048:",
    f"{(longitudes_modernbert > MAX_LENGTH).mean():.2%}"
)



Número de ejemplos (parejas en ambos órdenes): 8902
Ejemplos con más de 512 tokens: 2686
Porcentaje con más de 512 tokens: 30.17%
Longitud mediana: 416.0
Longitud máxima: 1640
Ejemplos con más de 2048 tokens: 0
Porcentaje truncado con 2048: 0.00%


#### Preparación de los datos para ModernBERT

Para cada comparación, el modelo recibe dos bloques de texto: el primero contiene la pregunta junto con una de las respuestas candidatas y el segundo contiene la otra respuesta. Cada pareja se presenta en ambos órdenes.

In [ ]:
def preparar_dataset_modernbert(
    pares,
    entrenamiento=True
):
    columnas = [
        "texto_a",
        "texto_b"
    ]

    if entrenamiento:
        columnas.append("label")

    dataset = Dataset.from_pandas(
        pares[columnas],
        preserve_index=False
    )

    def tokenizar(batch):
        return tokenizer(
            batch["texto_a"],
            batch["texto_b"],
            truncation=True,
            max_length=MAX_LENGTH
        )

    dataset = dataset.map(
        tokenizar,
        batched=True,
        remove_columns=[
            "texto_a",
            "texto_b"
        ]
    )

    if entrenamiento:
        dataset = dataset.rename_column(
            "label",
            "labels"
        )

    return dataset

#### Selección de la respuesta top-1

Durante la evaluación, ModernBERT compara todas las respuestas candidatas de una misma pregunta por parejas.

Cada respuesta acumula victorias y probabilidades de preferencia. La respuesta con mayor número de victorias se selecciona como top-1 y, en caso de empate, se utiliza la puntuación acumulada como criterio de desempate.

En esta fase el modelo recibe únicamente los textos. El target, definido previamente a partir de los upvotes, se utiliza después para comprobar si la respuesta seleccionada es correcta y calcular Accuracy@1.

In [ ]:
def crear_pares_prediccion(datos):

    datos_eval = (
        datos.reset_index(drop=True)
        .copy()
    )

    datos_eval["row_id"] = np.arange(
        len(datos_eval)
    )

    ejemplos = []
    pair_id = 0

    for question_id, grupo in datos_eval.groupby(
        "questionID"
    ):

        filas = list(
            grupo.itertuples(index=False)
        )

        for respuesta_1, respuesta_2 in combinations(
            filas, 2
        ):

            pregunta = str(
                respuesta_1.questionText
            )

            texto_1 = str(
                respuesta_1.answerText
            )

            texto_2 = str(
                respuesta_2.answerText
            )

            # Orden A-B
            ejemplos.append({
                "pair_id": pair_id,
                "questionID": question_id,
                "row_i": respuesta_1.row_id,
                "row_j": respuesta_2.row_id,
                "orden": "ij",
                "texto_a": (
                    f"Question: {pregunta}\n"
                    f"Response A: {texto_1}"
                ),
                "texto_b": (
                    f"Response B: {texto_2}"
                )
            })

            # Orden B-A
            ejemplos.append({
                "pair_id": pair_id,
                "questionID": question_id,
                "row_i": respuesta_1.row_id,
                "row_j": respuesta_2.row_id,
                "orden": "ji",
                "texto_a": (
                    f"Question: {pregunta}\n"
                    f"Response A: {texto_2}"
                ),
                "texto_b": (
                    f"Response B: {texto_1}"
                )
            })

            pair_id += 1

    return (
        datos_eval,
        pd.DataFrame(ejemplos)
    )

La siguiente función compara las respuestas, selecciona la respuesta top-1 de cada pregunta y, finalmente, utiliza el target, definido previamente a partir de los upvotes, para comprobar si la predicción fue correcta y calcular Accuracy@1.

In [ ]:
def evaluar_torneo(trainer, datos):

    datos_eval, pares = (
        crear_pares_prediccion(datos)
    )

    dataset = preparar_dataset_modernbert(
    pares,
    entrenamiento=False
    )

    predicciones = trainer.predict(
        dataset
    )

    logits = torch.tensor(
        predicciones.predictions
    )

    probabilidades = torch.softmax(
        logits,
        dim=1
    ).numpy()

    pares["prob_primera"] = (
        probabilidades[:, 0]
    )

    pares["prob_segunda"] = (
        probabilidades[:, 1]
    )

    victorias = {
        row_id: 0.0
        for row_id in datos_eval["row_id"]
    }

    puntuaciones = {
        row_id: 0.0
        for row_id in datos_eval["row_id"]
    }

    for _, grupo in pares.groupby(
        "pair_id"
    ):

        orden_ij = grupo[
            grupo["orden"] == "ij"
        ].iloc[0]

        orden_ji = grupo[
            grupo["orden"] == "ji"
        ].iloc[0]

        row_i = orden_ij["row_i"]
        row_j = orden_ij["row_j"]

        prob_i = (
            orden_ij["prob_primera"]
            + orden_ji["prob_segunda"]
        ) / 2

        prob_j = 1 - prob_i

        puntuaciones[row_i] += prob_i
        puntuaciones[row_j] += prob_j

        if prob_i > prob_j:
            victorias[row_i] += 1

        elif prob_j > prob_i:
            victorias[row_j] += 1

        else:
            victorias[row_i] += 0.5
            victorias[row_j] += 0.5

    ganadores = []

    for question_id, grupo in datos_eval.groupby(
        "questionID"
    ):

        grupo = grupo.copy()

        grupo["victorias"] = (
            grupo["row_id"]
            .map(victorias)
        )

        grupo["puntuacion"] = (
            grupo["row_id"]
            .map(puntuaciones)
        )

        ganador = (
            grupo.sort_values(
                ["victorias", "puntuacion"],
                ascending=[False, False]
            )
            .iloc[0]
        )

        ganadores.append(ganador)

    ganadores = pd.DataFrame(
        ganadores
    )

    accuracy = (
        ganadores["target"].mean()
    )

    return accuracy, ganadores

#### Validación cruzada con los 5 folds

Se realiza una validación cruzada de 5 folds utilizando exactamente las mismas particiones empleadas en el experimento con BERT.

En cada fold se inicializa un nuevo modelo ModernBERT desde sus pesos preentrenados y se entrena únicamente con las preguntas correspondientes al conjunto de entrenamiento de ese fold.

In [ ]:
accuracies_modernbert_cv = []

for fold, (train_idx, val_idx) in enumerate(
    cv_splits_equilibrados,
    start=1
):
    print(f"\n===== FOLD {fold} =====")

    train_fold = (
        train_equilibrado
        .iloc[train_idx]
        .copy()
    )

    val_fold = (
        train_equilibrado
        .iloc[val_idx]
        .copy()
    )

    print(
        "Preguntas train:",
        train_fold["questionID"].nunique()
    )

    print(
        "Preguntas validación:",
        val_fold["questionID"].nunique()
    )

    pares_train_fold = crear_pares_entrenamiento(
        train_fold
    )

    print(
        "Pares de entrenamiento:",
        len(pares_train_fold)
    )

    dataset_train_fold = preparar_dataset_modernbert(
        pares_train_fold,
        entrenamiento=True
    )

    set_seed(27)

    modelo_fold = (
        AutoModelForSequenceClassification
        .from_pretrained(
            MODEL_NAME,
            num_labels=2
        )
    )

    training_args = TrainingArguments(
        output_dir=f"./outputs/modernbert_fold_{fold}",
        num_train_epochs=2,
        per_device_train_batch_size=8,
        learning_rate=2e-5,
        weight_decay=0.01,
        save_strategy="no",
        logging_strategy="epoch",
        report_to="none",
        seed=27,
        data_seed=27
    )

    trainer_fold = Trainer(
        model=modelo_fold,
        args=training_args,
        train_dataset=dataset_train_fold,
        data_collator=data_collator
    )

    trainer_fold.train()

    accuracy_fold, _ = evaluar_torneo(
        trainer_fold,
        val_fold
    )

    accuracies_modernbert_cv.append(
        accuracy_fold
    )

    print(
        f"Accuracy@1 fold {fold}: "
        f"{accuracy_fold:.2%}"
    )

    del modelo_fold
    del trainer_fold
    del dataset_train_fold

    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()


===== FOLD 1 =====
Preguntas train: 176
Preguntas validación: 44
Pares de entrenamiento: 7982


Loading weights: 100%|█████████████████████| 136/136 [00:00<00:00, 3641.21it/s]
[transformers] ModernBertForSequenceClassification LOAD REPORT from: answerdotai/ModernBERT-base
Key               | Status     | 
------------------+------------+-
decoder.bias      | UNEXPECTED | 
classifier.bias   | MISSING    | 
classifier.weight | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Step,Training Loss
998,0.465283
1996,0.160671


Map: 100%|████████████████████████| 4378/4378 [00:00<00:00, 4563.99 examples/s]


Accuracy@1 fold 1: 45.45%

===== FOLD 2 =====
Preguntas train: 176
Preguntas validación: 44
Pares de entrenamiento: 8024


Loading weights: 100%|█████████████████████| 136/136 [00:00<00:00, 2468.45it/s]
[transformers] ModernBertForSequenceClassification LOAD REPORT from: answerdotai/ModernBERT-base
Key               | Status     | 
------------------+------------+-
decoder.bias      | UNEXPECTED | 
classifier.bias   | MISSING    | 
classifier.weight | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


... (11 outputs left out)

Loading weights: 100%|█████████████████████| 136/136 [00:00<00:00, 2505.54it/s]
[transformers] ModernBertForSequenceClassification LOAD REPORT from: answerdotai/ModernBERT-base
Key               | Status     | 
------------------+------------+-
decoder.bias      | UNEXPECTED | 
classifier.bias   | MISSING    | 
classifier.weight | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Step,Training Loss
764,0.485961
1528,0.273530


Map: 100%|████████████████████████| 9072/9072 [00:01<00:00, 5646.22 examples/s]


Accuracy@1 fold 5: 43.18%


Resumen de la validación cruzada:

In [ ]:
print(
    "Accuracy@1 media CV:",
    f"{np.mean(accuracies_modernbert_cv):.2%}"
)

print(
    "Desviación estándar:",
    f"{np.std(accuracies_modernbert_cv):.2%}"
)

Accuracy@1 media CV: 45.91%
Desviación estándar: 3.34%


ModernBERT obtiene una Accuracy@1 media del 45,91 %, frente al 44,09 % de BERT, una mejora de 1,82 puntos porcentuales. Además, la desviación estándar disminuye de 5,10 % a 3,34 %, lo que indica un rendimiento más estable entre folds. Por tanto, en validación cruzada ModernBERT muestra una mejora moderada respecto a BERT, aunque será necesario comprobar el resultado en test antes de sacar una conclusión final.

#### Entrenamiento final con todo train

Una vez completada la validación cruzada, se entrena un modelo ModernBERT final utilizando todas las preguntas del conjunto de entrenamiento. Se mantiene la misma configuración utilizada durante la validación cruzada.

In [ ]:
pares_train_final = crear_pares_entrenamiento(
    train_equilibrado
)

print(
    "Preguntas de entrenamiento:",
    train_equilibrado["questionID"].nunique()
)

print(
    "Ejemplos de entrenamiento (parejas en ambos órdenes):",
    len(pares_train_final)
)

Preguntas de entrenamiento: 220
Ejemplos de entrenamiento (parejas en ambos órdenes): 8902


In [ ]:
dataset_train_final = preparar_dataset_modernbert(
    pares_train_final,
    entrenamiento=True
)

set_seed(27)

modelo_modernbert_final = (
    AutoModelForSequenceClassification
    .from_pretrained(
        MODEL_NAME,
        num_labels=2
    )
)

training_args_final = TrainingArguments(
    output_dir="./outputs/modernbert_final",
    num_train_epochs=2,
    per_device_train_batch_size=8,
    learning_rate=2e-5,
    weight_decay=0.01,
    save_strategy="no",
    logging_strategy="epoch",
    report_to="none",
    seed=27,
    data_seed=27
)

trainer_modernbert_final = Trainer(
    model=modelo_modernbert_final,
    args=training_args_final,
    train_dataset=dataset_train_final,
    data_collator=data_collator
)

trainer_modernbert_final.train()

Loading weights: 100%|█████████████████████| 136/136 [00:00<00:00, 3094.12it/s]
[transformers] ModernBertForSequenceClassification LOAD REPORT from: answerdotai/ModernBERT-base
Key               | Status     | 
------------------+------------+-
decoder.bias      | UNEXPECTED | 
classifier.bias   | MISSING    | 
classifier.weight | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Step,Training Loss
1113,0.523552
2226,0.205591


TrainOutput(global_step=2226, training_loss=0.3645714676712187, metrics={'train_runtime': 1167.9579, 'train_samples_per_second': 15.244, 'train_steps_per_second': 1.906, 'total_flos': 9540289259798136.0, 'train_loss': 0.3645714676712187, 'epoch': 2.0})

#### Evaluación final en test

Finalmente, el modelo entrenado con todo el conjunto de entrenamiento se evalúa sobre el mismo conjunto de test utilizado en el experimento con BERT.

De esta forma, ambos modelos pueden compararse sobre exactamente las mismas preguntas.

In [ ]:
accuracy_modernbert_test, ganadores_modernbert = (
    evaluar_torneo(
        trainer_modernbert_final,
        test_equilibrado
    )
)

print(
    "ModernBERT pairwise "
    "- Accuracy@1 test:",
    f"{accuracy_modernbert_test:.2%}"
)

Map: 100%|████████████████████████| 3238/3238 [00:00<00:00, 3982.51 examples/s]


ModernBERT pairwise - Accuracy@1 test: 33.93%


Tabla resumen:

In [ ]:
resultados_modernbert = pd.DataFrame({
    "Métrica": [
        "Accuracy@1 media CV",
        "Desviación estándar CV",
        "Accuracy@1 test"
    ],
    "Resultado (%)": [
        np.mean(
            accuracies_modernbert_cv
        ) * 100,

        np.std(
            accuracies_modernbert_cv
        ) * 100,

        accuracy_modernbert_test * 100
    ]
})

display(
    resultados_modernbert.round(2)
)

,Métrica,Resultado (%)
0,Accuracy@1 media CV,45.91
1,Desviación estándar CV,3.34
2,Accuracy@1 test,33.93


ModernBERT obtiene una Accuracy@1 media del 45,91 % ± 3,34 % en validación cruzada y un 33,93 % en test. Frente a BERT, mejora ligeramente la media de validación (pasa del 44,09 % al 45,91 %) y reduce la variabilidad entre folds (del 5,10 % al 3,34 %). Sin embargo, en el mismo conjunto de test su rendimiento es inferior (33,93 % frente al 37,50 % de BERT). Por tanto, aunque ModernBERT muestra resultados algo mejores y más estables en validación, no mejora el rendimiento final en test.

#### Comparación con BERT

Finalmente, se comparan los resultados de ModernBERT con los obtenidos previamente mediante `bert-base-uncased`.

Ambos experimentos utilizan las mismas particiones, el mismo procedimiento pairwise y la misma estrategia de evaluación. La principal diferencia es la capacidad de ModernBERT para procesar secuencias más largas, lo que elimina el truncado observado con BERT.

In [ ]:
resultados_comparacion = pd.DataFrame({
    "Modelo": [
        "BERT",
        "ModernBERT"
    ],
    "Accuracy@1 CV (%)": [
        44.09,
        np.mean(
            accuracies_modernbert_cv
        ) * 100
    ],
    "Desv. estándar CV (%)": [
        5.10,
        np.std(
            accuracies_modernbert_cv
        ) * 100
    ],
    "Accuracy@1 test (%)": [
        37.50,
        accuracy_modernbert_test * 100
    ]
})

display(
    resultados_comparacion.round(2)
)

,Modelo,Accuracy@1 CV (%),Desv. estándar CV (%),Accuracy@1 test (%)
0,BERT,44.09,5.10,37.50
1,ModernBERT,45.91,3.34,33.93


Estos resultados indican que reducir el truncado, por sí solo, no garantiza una mejora del rendimiento del modelo.